# 03A.05A — Ablação controlada das variáveis de trajetória

## Como executar no Google Colab

1. Abra este arquivo no Colab, em um ambiente novo com **CPU**.
2. Escolha **Ambiente de execução → Executar tudo** e autorize o acesso ao Drive.
3. Ao terminar, salve o notebook com as saídas e baixe uma cópia em **Arquivo → Fazer download → Fazer download do .ipynb**.

A pasta esperada é `/content/drive/MyDrive/co2-cems-thermal-modeling`, a mesma dos notebooks anteriores. Se ela estiver em outro lugar, ajuste somente `PASTA_PROJETO` na primeira célula de código.

Este notebook usa os arquivos produzidos pelo 03A.05 já executado. Não precisa de chave EPA, não baixa novos dados e não altera os notebooks nem os resultados anteriores. Cada execução salva seus resultados em uma subpasta datada de `outputs/03A05A_ablacao_controlada`.

**Controle obrigatório:** antes de retirar variáveis, o modelo completo precisa reproduzir as previsões recebidas, com diferença máxima de 10⁻⁶ short ton. Se essa conferência falhar, as variantes ficam bloqueadas. Preserve o erro e envie o notebook com suas saídas.

## Pergunta e protocolo

Experimento complementar ao 03A.05 executado em 29/10/2026. São usados os mesmos dados, a mesma ordem das linhas, semente, divisão temporal e hiperparâmetros; o número de árvores é o já selecionado para o cenário B. Não há novo ajuste de hiperparâmetros.

Zhang et al. (2026), DOI 10.1109/CCSSTA69471.2026.11635773, motivam o diagnóstico de redundância. O artigo usa importância por XGBoost, cotovelo, Pearson com |r| > 0,6 e LSTM. Aqui mantemos XGBoost e seu protocolo original. O alerta |r| ≥ 0,90 e as variantes abaixo são adaptações do presente estudo.

As hipóteses vêm do treino e das identidades das variáveis:

1. Retirar a média das três horas anteriores, altamente correlacionada com a carga defasada.
2. Retirar ΔP, reconstruível de P(t) e P(t−1).
3. Retirar ΔP e |ΔP|, ambos reconstruíveis das cargas preservadas.

Correlação ou dependência algébrica não garantem que uma variável seja inútil para árvores. Essa é a hipótese testada. O teste histórico já foi reutilizado no desenvolvimento; a comparação é exploratória e não demonstra equivalência estatística.

In [4]:
# Altere somente esta linha se a pasta do projeto estiver em outro local do Drive.
PASTA_PROJETO = "/content/drive/MyDrive/co2-cems-thermal-modeling"

from pathlib import Path
from datetime import datetime, timezone
import sys, subprocess, importlib.metadata

# Instalar antes de importar XGBoost. Um ambiente novo evita versões já carregadas.
XGBOOST_VERSION = "3.4.1"
if "xgboost" in sys.modules and sys.modules["xgboost"].__version__ != XGBOOST_VERSION:
    raise RuntimeError("Reinicie a sessão do Colab e execute tudo novamente: outra versão de XGBoost já foi carregada.")
try:
    installed_version = importlib.metadata.version("xgboost")
except importlib.metadata.PackageNotFoundError:
    installed_version = None
if installed_version != XGBOOST_VERSION:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost==" + XGBOOST_VERSION])

import os, json, hashlib, ast, random, platform
import numpy as np
import pandas as pd
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

# A execução local é opcional; no Colab, o caminho padrão é o Drive.
project_override = os.environ.get("CO2_PROJECT_ROOT")
if project_override:
    SOURCE = Path(project_override).expanduser().resolve()
else:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("Abra este notebook no Google Colab ou defina CO2_PROJECT_ROOT para execução local.") from exc
    drive.mount("/content/drive")
    SOURCE = Path(PASTA_PROJETO)

assert SOURCE.is_dir(), f"Pasta não encontrada: {SOURCE}. Confira PASTA_PROJETO."
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
OUT = SOURCE / "outputs" / "03A05A_ablacao_controlada" / run_id
OUT.mkdir(parents=True, exist_ok=False)
print("Resultados desta execução:", OUT)
BASELINE_OK = False
DATA = SOURCE / 'data/interim/03A05_trajectory_history'
TABLES = SOURCE / 'outputs/tables'
TARGET = 'CO2 Mass (short tons)'
FEATURES = ['load_current_mw','load_lag_1h','delta_load_1h','abs_delta_load_1h',
            'load_mean_previous_3h','operating_time_lag_1h','hours_since_startup']
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

input_paths = [DATA/'03A05_treino_final_A_B.parquet',
               DATA/'03A05_teste_nov_dez_2025_A_B.parquet',
               DATA/'03A05_previsoes_pareadas_com_contextos.parquet',
               TABLES/'03A05_selecao_complexidade_A_B.csv',
               SOURCE/'notebooks/03A.05_trajetoria_operacional_historico_ampliado.ipynb']
missing = [str(p) for p in input_paths if not p.is_file()]
assert not missing, 'Arquivos do 03A.05 não encontrados: ' + repr(missing)
input_hashes = {str(p.relative_to(SOURCE)): digest(p) for p in input_paths}
train = pd.read_parquet(input_paths[0])
test = pd.read_parquet(input_paths[1])
received_predictions = pd.read_parquet(input_paths[2])
selection = pd.read_csv(input_paths[3])
source_nb = json.loads(input_paths[4].read_text(encoding='utf-8'))
first_code = next(''.join(c['source']) for c in source_nb['cells'] if c['cell_type']=='code')
tree = ast.parse(first_code)
params_node = next(n.value for n in tree.body if isinstance(n, ast.Assign)
                   and any(isinstance(t, ast.Name) and t.id=='XGB_PARAMS_SELECTION' for t in n.targets))
params = {}
for key, value in zip(params_node.keys, params_node.values):
    params[ast.literal_eval(key)] = SEED if isinstance(value, ast.Name) and value.id=='SEED' else ast.literal_eval(value)
params.pop('early_stopping_rounds')
params['n_estimators'] = int(selection.loc[selection['scenario']=='B','selected_trees'].iloc[0])
assert params['random_state']==SEED
assert params['n_estimators']==721, 'A seleção recebida para B deve ter 721 árvores.'
assert xgb.__version__=='3.4.1', 'Usar a mesma versão observada na execução recebida.'
assert train['timestamp'].is_monotonic_increasing and test['timestamp'].is_monotonic_increasing
assert train['timestamp'].is_unique and test['timestamp'].is_unique
assert train['timestamp'].max() < test['timestamp'].min()
assert set(train['timestamp'].dt.year)==set(range(2011,2023))|{2025}
assert test['timestamp'].min()==pd.Timestamp('2025-11-01')
assert test['timestamp'].max()==pd.Timestamp('2025-12-31 23:00')
assert len(train)==88686 and len(test)==1144
assert not train[FEATURES+[TARGET]].isna().any().any()
assert not test[FEATURES+[TARGET]].isna().any().any()
assert np.array_equal(test['timestamp'].to_numpy(), received_predictions['timestamp'].to_numpy())
assert np.array_equal(test[TARGET].to_numpy(), received_predictions[TARGET].to_numpy())
assert np.allclose(train['load_current_mw']-train['load_lag_1h'],train['delta_load_1h'],rtol=0,atol=1e-9)
assert np.allclose(train['delta_load_1h'].abs(),train['abs_delta_load_1h'],rtol=0,atol=1e-9)
corr = train[FEATURES].corr()
corr.to_csv(OUT/'correlacoes_treino.csv')
display(corr.round(6))
print('Amostra, ordem, alvos, separação temporal e identidades confirmados.')
print('Versão XGBoost:',xgb.__version__)
print('Hiperparâmetros fixos:',params)

environment = {
    'python': platform.python_version(), 'platform': platform.platform(),
    'xgboost': xgb.__version__, 'numpy': np.__version__, 'pandas': pd.__version__,
    'scikit_learn': importlib.metadata.version('scikit-learn'),
    'pyarrow': importlib.metadata.version('pyarrow'),
    'fixed_params': params, 'input_hashes': input_hashes, 'run_id_utc': run_id,
}
(OUT/'ambiente_e_insumos.json').write_text(json.dumps(environment, ensure_ascii=False, indent=2), encoding='utf-8')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Resultados desta execução: /content/drive/MyDrive/co2-cems-thermal-modeling/outputs/03A05A_ablacao_controlada/20261030T014212_748187Z


,load_current_mw,load_lag_1h,delta_load_1h,abs_delta_load_1h,load_mean_previous_3h,operating_time_lag_1h,hours_since_startup
load_current_mw,1.000000,0.975552,0.077272,-0.203380,0.941187,0.284424,0.194615
load_lag_1h,0.975552,1.000000,-0.143727,-0.182285,0.983568,0.281931,0.196708
delta_load_1h,0.077272,-0.143727,1.000000,-0.088858,-0.223928,0.001742,-0.016044
abs_delta_load_1h,-0.203380,-0.182285,-0.088858,1.000000,-0.182248,0.029332,-0.101489
load_mean_previous_3h,0.941187,0.983568,-0.223928,-0.182248,1.000000,0.276887,0.201164
operating_time_lag_1h,0.284424,0.281931,0.001742,0.029332,0.276887,1.000000,0.050114
hours_since_startup,0.194615,0.196708,-0.016044,-0.101489,0.201164,0.050114,1.000000


Amostra, ordem, alvos, separação temporal e identidades confirmados.
Versão XGBoost: 3.4.1
Hiperparâmetros fixos: {'objective': 'reg:squarederror', 'n_estimators': 721, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'eval_metric': 'rmse', 'random_state': 42, 'n_jobs': -1}


1303

## Reconstituição do modelo completo

Esta etapa condiciona a comparação: o modelo completo deve reproduzir as previsões
recebidas com diferença máxima de 10⁻⁶ short ton. Se falhar, a execução para antes
de testar variantes. O limiar é numérico, não um critério de equivalência entre modelos.

In [5]:
BASELINE_OK = False
baseline = XGBRegressor(**params)
baseline.fit(train[FEATURES], train[TARGET], verbose=False)
baseline_prediction = baseline.predict(test[FEATURES])
max_baseline_difference = float(np.max(np.abs(baseline_prediction-received_predictions['pred_B'].to_numpy())))
baseline_exact = bool(np.array_equal(baseline_prediction, received_predictions['pred_B'].to_numpy()))
baseline_check = {'max_absolute_prediction_difference':max_baseline_difference,
                  'exact_prediction_equality':baseline_exact,
                  'n_predictions':len(test), 'tolerance_absolute':1e-6}
baseline_check['passed'] = max_baseline_difference <= 1e-6
(OUT/'baseline_reproduction.json').write_text(json.dumps(baseline_check,indent=2),encoding='utf-8')
display(pd.DataFrame([baseline_check]))
assert max_baseline_difference<=1e-6, 'Baseline diferente: interromper e diagnosticar ambiente/protocolo.'

BASELINE_OK = bool(baseline_check['passed'])
print("Modelo completo reproduzido. As variantes estão liberadas.")


,max_absolute_prediction_difference,exact_prediction_equality,n_predictions,tolerance_absolute,passed
0,0.0,True,1144,0.000001,True


Modelo completo reproduzido. As variantes estão liberadas.


## Variantes e avaliação pareada

Todas as variantes usam as mesmas observações e os hiperparâmetros extraídos do
03A.05, incluindo as árvores selecionadas para B. A redução do número de colunas
altera a quantidade efetiva de candidatas sob `colsample_bytree=0.8`, embora seu
valor permaneça fixo; o efeito medido é o da representação dentro desse algoritmo.
Os contextos são os já usados no notebook original, com P90 calculado no treino.
Primeiras horas após startup e rampas podem se sobrepor.

In [6]:
assert BASELINE_OK, "Variantes bloqueadas: execute e aprove a conferência do modelo completo na célula anterior."
assert all(digest(SOURCE/relative)==value for relative,value in input_hashes.items()), "Os insumos mudaram durante a execução."
variants = {
    'B_completo':FEATURES,
    'B_sem_media3h':[f for f in FEATURES if f!='load_mean_previous_3h'],
    'B_sem_delta':[f for f in FEATURES if f!='delta_load_1h'],
    'B_sem_delta_e_absdelta':[f for f in FEATURES if f not in ['delta_load_1h','abs_delta_load_1h']],
}
ramp_threshold = float(train['abs_delta_load_1h'].quantile(0.90))
assert np.array_equal(test['abs_delta_load_1h'].ge(ramp_threshold),received_predictions['intense_ramp'])
masks = {'global':np.ones(len(test),dtype=bool),
         'primeiras_6h_startup':received_predictions['startup_first_6h'].to_numpy(dtype=bool),
         'rampas_intensas':received_predictions['intense_ramp'].to_numpy(dtype=bool),
         'startup_e_rampa':received_predictions['startup_and_ramp'].to_numpy(dtype=bool),
         'demais_horas':received_predictions['regular_other'].to_numpy(dtype=bool)}
def metrics(y, prediction):
    y=np.asarray(y,dtype=float); prediction=np.asarray(prediction,dtype=float)
    mae=float(mean_absolute_error(y,prediction))
    return {'MAE':mae,'RMSE':float(np.sqrt(mean_squared_error(y,prediction))),
            'R2':float(r2_score(y,prediction)) if len(y)>1 else float('nan'), 'NMAE_pct':100*mae/float(y.mean()),
            'MedianAE':float(np.median(np.abs(y-prediction))), 'Bias':float(np.mean(y-prediction))}
rows=[]
predictions=received_predictions[['timestamp',TARGET,'pred_A','pred_B','intense_ramp','startup_first_6h','regular_other']].copy()
for variant, features in variants.items():
    print('Executando:', variant, '| variáveis:', len(features), flush=True)
    if variant=='B_completo':
        model=baseline;prediction=baseline_prediction
    else:
        model=XGBRegressor(**params)
        model.fit(train[features],train[TARGET],verbose=False)
        prediction=model.predict(test[features])
    model.save_model(OUT/f'{variant}.json')
    predictions['pred_'+variant]=prediction
    for regime, mask in masks.items():
        if not mask.any():continue
        current=metrics(test[TARGET].to_numpy()[mask],prediction[mask])
        original=metrics(test[TARGET].to_numpy()[mask],baseline_prediction[mask])
        rows.append(dict(variant=variant,regime=regime,n_features=len(features),n_train=len(train),n_test=int(mask.sum()),
                         **current, MAE_difference_vs_B=current['MAE']-original['MAE'],
                         MAE_difference_pct_vs_B=100*(current['MAE']/original['MAE']-1),
                         RMSE_difference_vs_B=current['RMSE']-original['RMSE'],
                         RMSE_difference_pct_vs_B=100*(current['RMSE']/original['RMSE']-1),
                         R2_difference_vs_B=current['R2']-original['R2'],
                         NMAE_difference_pp_vs_B=current['NMAE_pct']-original['NMAE_pct']))
results=pd.DataFrame(rows)
results.to_csv(OUT/'ablacao_metricas.csv',index=False)
predictions.to_csv(OUT/'ablacao_previsoes.csv',index=False)
predictions.to_parquet(OUT/'ablacao_previsoes.parquet',index=False)
protocol={'fixed_params':params,'variants':variants,'n_train':len(train),'n_test':len(test),
          'train_start':str(train['timestamp'].min()),'train_end':str(train['timestamp'].max()),
          'test_start':str(test['timestamp'].min()),'test_end':str(test['timestamp'].max()),
          'ramp_threshold':ramp_threshold,'baseline_check':baseline_check,
          'input_hashes':input_hashes,'python':platform.python_version(),'platform':platform.platform(),
          'environment':environment,
          'xgboost':xgb.__version__,'numpy':np.__version__,'pandas':pd.__version__,
          'warning':'Comparação exploratória em teste histórico já reutilizado. Sem teste formal de equivalência.'}
assert all(digest(SOURCE/relative)==value for relative,value in input_hashes.items())
(OUT/'protocolo_ablacao.json').write_text(json.dumps(protocol,ensure_ascii=False,indent=2),encoding='utf-8')
display(results.round(6))
print('Concluído. MAE_difference_pct_vs_B negativo indica redução do MAE em relação ao B completo.')
print('Nenhum arquivo recebido foi modificado.')

print("Pasta dos resultados:", OUT)
print("Salve este notebook com as saídas e envie também ablacao_metricas.csv e protocolo_ablacao.json.")


Executando: B_completo | variáveis: 7
Executando: B_sem_media3h | variáveis: 6
Executando: B_sem_delta | variáveis: 6
Executando: B_sem_delta_e_absdelta | variáveis: 5


,variant,regime,n_features,n_train,n_test,MAE,RMSE,R2,NMAE_pct,MedianAE,Bias,MAE_difference_vs_B,MAE_difference_pct_vs_B,RMSE_difference_vs_B,RMSE_difference_pct_vs_B,R2_difference_vs_B,NMAE_difference_pp_vs_B
0,B_completo,global,7,88686,1144,21.171987,26.577858,0.982844,3.597365,17.855573,-2.550333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,B_completo,primeiras_6h_startup,7,88686,30,10.451707,13.165937,0.733215,22.913274,9.375541,-8.533026,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,B_completo,rampas_intensas,7,88686,258,18.549117,22.480886,0.980416,3.216592,16.575708,-3.588153,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,B_completo,startup_e_rampa,7,88686,1,25.190802,25.190802,NaN,15.304254,25.190802,-25.190802,0.000000,0.000000,0.000000,0.000000,NaN,0.000000
4,B_completo,demais_horas,7,88686,857,22.341564,28.026185,0.978499,3.658794,18.609265,-2.054886,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,B_sem_media3h,global,6,88686,1144,20.689602,25.858607,0.983760,3.515402,17.195514,-3.987032,-0.482385,-2.278410,-0.719251,-2.706206,0.000916,-0.081963
6,B_sem_media3h,primeiras_6h_startup,6,88686,30,10.239119,13.168128,0.733126,22.447218,9.031968,-8.244002,-0.212588,-2.033999,0.002191,0.016641,-0.000089,-0.466056
7,B_sem_media3h,rampas_intensas,6,88686,258,19.279615,23.197698,0.979147,3.343267,16.975812,-5.895560,0.730498,3.938181,0.716812,3.188539,-0.001269,0.126675
8,B_sem_media3h,startup_e_rampa,6,88686,1,29.154898,29.154898,NaN,17.712575,29.154898,-29.154898,3.964096,15.736284,3.964096,15.736284,NaN,2.408321
9,B_sem_media3h,demais_horas,6,88686,857,21.489785,26.935364,0.980140,3.519302,17.704089,-3.292818,-0.851779,-3.812531,-1.090822,-3.892153,0.001641,-0.139493


Concluído. MAE_difference_pct_vs_B negativo indica redução do MAE em relação ao B completo.
Nenhum arquivo recebido foi modificado.
Pasta dos resultados: /content/drive/MyDrive/co2-cems-thermal-modeling/outputs/03A05A_ablacao_controlada/20261030T014212_748187Z
Salve este notebook com as saídas e envie também ablacao_metricas.csv e protocolo_ablacao.json.


## Como interpretar

Os sinais e magnitudes das diferenças descrevem este teste histórico. Redução de
MAE ou RMSE não demonstra benefício em uma futura amostra independente. Um erro
parecido com menos variáveis não constitui equivalência estatística. As variantes
com piora também permanecem registradas. O conjunto principal de atributos do
artigo e dos notebooks recebidos não é automaticamente substituído.

## O que enviar para inspeção

Após a conclusão, envie a cópia deste `.ipynb` **com as saídas salvas**, junto de `ablacao_metricas.csv` e `protocolo_ablacao.json`, na pasta de resultados mostrada acima. Os demais arquivos também permanecem no Drive para auditoria. Se houver erro, envie o notebook com a mensagem completa; não remova a conferência do modelo completo.


## Origem dos procedimentos e adaptações

Zhang et al. (2026) motivam a investigação de redundância, com dados de um minuto em operação estável e um estimador LSTM. Neste notebook, as variantes, o limiar descritivo de correlação e a manutenção de 721 árvores são decisões do projeto. Não se reproduz a seleção em duas etapas do artigo nem se presume que uma identidade algébrica torne uma variável inútil para árvores.

Friedman (2001) e Chen e Guestrin (2016) fundamentam boosting e XGBoost. Hyndman e Koehler (2006) discutem MAE/RMSE; o cálculo das métricas segue Scikit-learn (2025, documentação 1.6), correspondente ao scikit-learn 1.6.1 registrado nesta execução. A convenção de viés, o denominador do NMAE e os contextos são escolhas do protocolo original.

Cawley e Talbot (2010) fundamentam a cautela com seleção e avaliação no mesmo percurso de desenvolvimento. Cerqueira, Torgo e Mozetič (2020) fundamentam o cuidado com dependência temporal; não validam automaticamente este teste nem um ganho futuro. As saídas e todo o código executado permanecem preservados.


## Complemento bibliográfico e avaliação separada

A retirada de atributos é um diagnóstico da representação, tema discutido por Grinsztajn, Oyallon e Varoquaux (2022), sem exclusão automática por correlação. A execução recebida preservou parâmetros, 721 árvores, semente e amostras e reproduziu exatamente B completo. A análise pareada de perdas, motivada por Diebold e Mariano (1994), está no complemento 03A.05B executado separadamente. Seus blocos de calendário remetem ao princípio geral de Künsch (1989), com limites exploratórios e sem garantia de cobertura ou correção da seleção retrospectiva. Nenhum DM/HAC é aplicado; os resultados originais e o ZIP recebido são preservados.

## Referências

CAWLEY, G. C.; TALBOT, N. L. C. On Over-fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation. Journal of Machine Learning Research, [s. l.], v. 11, n. 70, p. 2079-2107, 2010. Disponível em: https://jmlr.org/papers/v11/cawley10a.html. Acesso em: 29 out. 2026.

CERQUEIRA, V.; TORGO, L.; MOZETIČ, I. Evaluating time series forecasting models: an empirical study on performance estimation methods. Machine Learning, [s. l.], v. 109, n. 11, p. 1997-2028, 2020. DOI: https://doi.org/10.1007/s10994-020-05910-7. Disponível em: https://link.springer.com/article/10.1007/s10994-020-05910-7. Acesso em: 29 out. 2026.

CHEN, T.; GUESTRIN, C. XGBoost: A Scalable Tree Boosting System. In: ACM SIGKDD INTERNATIONAL CONFERENCE ON KNOWLEDGE DISCOVERY AND DATA MINING, 22., 2016, San Francisco. Proceedings [...]. New York: ACM, 2016. p. 785-794. DOI: https://doi.org/10.1145/2939672.2939785. Disponível em: https://www.kdd.org/kdd2016/papers/files/rfp0697-chenAemb.pdf. Versão dos autores consultada: arXiv:1603.02754, https://arxiv.org/abs/1603.02754. Acesso em: 29 out. 2026.

DIEBOLD, F. X.; MARIANO, R. S. Comparing Predictive Accuracy. Cambridge, MA: National Bureau of Economic Research, nov. 1994. Technical Working Paper 169. DOI: https://doi.org/10.3386/t0169. Disponível em: https://www.nber.org/papers/t0169. Acesso em: 29 out. 2026.

FRIEDMAN, J. H. Greedy function approximation: A gradient boosting machine. The Annals of Statistics, [s. l.], v. 29, n. 5, p. 1189-1232, 2001. DOI: https://doi.org/10.1214/aos/1013203451. Disponível em: https://projecteuclid.org/journals/annals-of-statistics/volume-29/issue-5/Greedy-function-approximation-A-gradient-boosting-machine/10.1214/aos/1013203451.full. Acesso em: 29 out. 2026.

GRINSZTAJN, L.; OYALLON, E.; VAROQUAUX, G. Why do tree-based models still outperform deep learning on typical tabular data? Advances in Neural Information Processing Systems, v. 35, Datasets and Benchmarks Track, 2022. DOI: https://doi.org/10.52202/068431-0037. Disponível em: https://proceedings.neurips.cc/paper_files/paper/2022/hash/0378c7692da36807bdec87ab043cdadc-Abstract-Datasets_and_Benchmarks.html. Acesso em: 29 out. 2026.

HYNDMAN, R. J.; KOEHLER, A. B. Another look at measures of forecast accuracy. International Journal of Forecasting, [s. l.], v. 22, n. 4, p. 679-688, 2006. DOI: https://doi.org/10.1016/j.ijforecast.2006.03.001. Disponível em: https://robjhyndman.com/publications/another-look-at-measures-of-forecast-accuracy/. Manuscrito dos autores consultado, de 2 nov. 2005: https://robjhyndman.com/papers/mase.pdf, correspondente à publicação de 2006. Acesso em: 29 out. 2026.

KÜNSCH, H. R. The Jackknife and the Bootstrap for General Stationary Observations. The Annals of Statistics, v. 17, n. 3, p. 1217-1241, 1989. DOI: https://doi.org/10.1214/aos/1176347265. Acesso em: 29 out. 2026.

SCIKIT-LEARN. User Guide. Version 1.6.1. [S. l.]: The scikit-learn developers, [2025]. Seção Regression metrics. Disponível em: https://scikit-learn.org/1.6/modules/model_evaluation.html#regression-metrics. Acesso em: 29 out. 2026.

ZHANG, K.; WU, Z.; LIU, Z.; WANG, J. Dynamic Prediction of CO₂ Emissions from Coal-Fired Power Plants Based on XGBoost Hybrid Feature Selection and LSTM. In: 2026 IEEE 27th China Conference on System Simulation Technology and its Applications (CCSSTA). [S. l.]: IEEE, 2026. p. 865-870. DOI: https://doi.org/10.1109/CCSSTA69471.2026.11635773. Acesso em: 29 out. 2026.
